# `CarlemanBasis`

The plain monomial basis, and the one the examples use most.

## 1. The basis functions

Monomials in the state:

$$y_{\boldsymbol\alpha}(u)=\prod_i u_i^{\alpha_i},
\qquad |\boldsymbol\alpha|=\sum_i\alpha_i\le K,$$

for a truncation degree $K$. On $N$ sites there are $\binom{N+K}{K}$ of them.

They are **not** orthogonal, and a monomial is not square-integrable on $\mathbb{R}^{N}$. So there is no
inner product to project with. `Liouville` and `KoopmanVonNeumann` need a square-integrable basis,
so `OperatorMatrix` raises `IncompatibleBasisError` on this one ([`03_linearization`](../03_linearization.ipynb)).

## 2. The state class: `CarlemanState`

A `CarlemanState` prints its `(site, power)` label.

In [1]:
import sympy as sp

from lin_ode.basis import CarlemanBasis

basis = CarlemanBasis(n_vars=3, degree=3)

print("basis[()]               ", basis[()])
print("basis[(1, 2)]           ", basis[(1, 2)])
print("basis[((0, 1), (2, 1))] ", basis[((0, 1), (2, 1))])

basis[()]                CarlemanState()
basis[(1, 2)]            CarlemanState((1, 2))
basis[((0, 1), (2, 1))]  CarlemanState((0, 1), (2, 1))


## 3. The basis class: `CarlemanBasis`

`CarlemanBasis(n_vars, degree)` takes two arguments and no others. The states follow from the site
count and the degree.

In [2]:
for degree in (2, 3, 4):
    candidate = CarlemanBasis(3, degree)
    print(f"degree {degree}:  len={len(candidate):3}  n_dof={candidate.n_dof}")

degree 2:  len= 10  n_dof=3
degree 3:  len= 20  n_dof=3
degree 4:  len= 35  n_dof=3


## 4. Indexing

Structured `(site, power)` tuples address a basis state, and a flat index enumerates them. Both
directions are available. An out-of-basis state has no flat index.

In [3]:
basis = CarlemanBasis(n_vars=2, degree=2)

for flat in range(len(basis)):
    structured = basis.structured_idx_of_flat_index(flat)
    print(f"  flat {flat}  <->  {str(structured):20}  {basis[structured]}")

print()
print("flat_index_of_structured_index((1, 2)) ", basis.flat_index_of_structured_index((1, 2)))
print("flat_index_of_basis_function(u_1^2)    ", basis.flat_index_of_basis_function(basis[(1, 2)]))
print("out of basis -> no flat index          ", basis.flat_index_of_basis_function(basis[(1, 3)]))

  flat 0  <->  ()                    CarlemanState()
  flat 1  <->  ((0, 1),)             CarlemanState((0, 1))
  flat 2  <->  ((1, 1),)             CarlemanState((1, 1))
  flat 3  <->  ((0, 2),)             CarlemanState((0, 2))
  flat 4  <->  ((0, 1), (1, 1))      CarlemanState((0, 1), (1, 1))
  flat 5  <->  ((1, 2),)             CarlemanState((1, 2))

flat_index_of_structured_index((1, 2))  5
flat_index_of_basis_function(u_1^2)     5
out of basis -> no flat index           None


## 5. The computation rules

:::{note}
Sections 5 to 7 are how the library produces a value, not something you call.
`OperatorMatrix` and `Vector` apply these rules for you. Read them when you are adding a
basis of your own, or when an element is not the number you expected.
:::

A linearized operator needs two things from a basis: what multiplication by $u_s$ does to a basis
state, and what $\partial/\partial u_s$ does.

Both are index arithmetic here, which is why an element costs the same at any grid size.

On monomials both are one term:

$$U_s\,y_{\boldsymbol\alpha}=y_{\boldsymbol\alpha+\mathbf e_s},
\qquad
K_s\,y_{\boldsymbol\alpha}=\alpha_s\,y_{\boldsymbol\alpha-\mathbf e_s}.$$

$U$ raises the power at site $s$. $K$ lowers it and leaves the old power behind as a factor.

In [4]:
from lin_ode.basis import CarlemanKRule, CarlemanURule

print("U on (site 0, n=2):", CarlemanURule()(sp.Integer(0), sp.Integer(2)))
print("K on (site 0, n=2):", CarlemanKRule()(sp.Integer(0), sp.Integer(2)))

U on (site 0, n=2): CarlemanState((0, 3))
K on (site 0, n=2): 2*CarlemanState((0, 1))


That pair is the entire computation rule. Apply $K_s$ to a monomial and the generator
$\mathcal K=\sum_s f_s(u)\,K_s$ gives

$$\mathcal K\,y_{\boldsymbol\alpha}=\sum_s f_s(u)\,K_s\,y_{\boldsymbol\alpha}
=\sum_s \alpha_s\,f_s(u)\,y_{\boldsymbol\alpha-\mathbf e_s},$$

a sum of polynomials. The coefficient of a target state $y_m$ is the coefficient of that monomial.
No integral and no orthogonality are involved. The operation is reading a coefficient off a
polynomial.

[`05_matrix`](../05_matrix.ipynb) works a whole column out this way.

## 6. Truncation

`truncation_rule` returns a state's representation inside the basis. For a state in the basis that
is the state itself. Above the degree it is `0`, and the state is gone.

In [5]:
basis = CarlemanBasis(n_vars=2, degree=2)


def factors(address: tuple) -> tuple:
    """(site, power) tuples, for either a single factor or a tuple of them."""
    return (address,) if isinstance(address[0], int) else address


for address in [(0, 1), (0, 2), (0, 3), ((0, 2), (1, 1))]:
    total = sum(power for _, power in factors(address))
    print(f"  {str(address):18} degree {total}  ->  {basis.truncation_rule(*factors(address))}")

  (0, 1)             degree 1  ->  CarlemanState((0, 1))
  (0, 2)             degree 2  ->  CarlemanState((0, 2))
  (0, 3)             degree 3  ->  0
  ((0, 2), (1, 1))   degree 3  ->  0


## 7. Expressions after truncation

`truncate_expression` applies the rule to a whole expression. For a dropping rule it is only
`sp.expand`. Projection discards the out-of-basis terms anyway, so there is nothing to substitute.

In [6]:
u = sp.IndexedBase("u")
basis = CarlemanBasis(n_vars=2, degree=2)

print("u_0**3            ->", basis.truncate_expression(u[0] ** 3, u))
print("u_0**2 * u_1      ->", basis.truncate_expression(u[0] ** 2 * u[1], u))
print()
print("but the states themselves are gone:")
print("  truncation_rule((0, 3))       =", basis.truncation_rule((0, 3)))
print("  truncation_rule((0, 2), (1, 1)) =", basis.truncation_rule((0, 2), (1, 1)))

u_0**3            -> u[0]**3
u_0**2 * u_1      -> u[0]**2*u[1]

but the states themselves are gone:
  truncation_rule((0, 3))       = 0
  truncation_rule((0, 2), (1, 1)) = 0


The states above the degree are simply absent, and any coupling that needed them is lost. That is
the truncation error. For what it costs a solution, see [`05_matrix`](../05_matrix.ipynb) and
[`08_end_to_end`](../08_end_to_end.ipynb).

---

[← 3. Bases](index.ipynb) · [Notebook index](../../README.md) · [PSCarlemanBasis →](psc_carleman.ipynb)